# Python + MySQL Connectivity

## What you will learn

1. How Python communicates with a MySQL server
2. Installing and importing the connector
3. Creating a database and tables
4. Opening and closing connections
5. Using cursors and executing SQL
6. Reading, inserting, updating, and deleting records
7. Parameterized queries and SQL injection prevention
8. Transactions, commit, and rollback
9. Handling errors safely
10. Building reusable connection functions and CRUD examples

> **Before running:** Start your MySQL server. Update the connection settings in the configuration cell below. The examples use a database named `python_mysql_demo` and a table named `employees`.

## 1. How the connection works

Your Python program does not directly manipulate MySQL's files. It sends SQL statements to a running MySQL server through a connector library.

**Flow:**

`Python application → mysql-connector-python → MySQL Server → Database`

Important terms:

- **MySQL server:** The database service running on your machine or another host.
- **Database:** A logical container for tables and other database objects.
- **Table:** Stores rows and columns.
- **Connection:** A session between Python and the MySQL server.
- **Cursor:** An object used to execute SQL statements and fetch results.
- **Transaction:** A group of database operations that can be committed or rolled back.

## 2. Install the connector

Run this command in a terminal or in a notebook cell:

```bash
pip install mysql-connector-python
```

If you use a virtual environment, activate it before installing the package.

To check the installed package version:

In [ ]:
from importlib.metadata import version

print("mysql-connector-python:", version("mysql-connector-python"))

## 3. Import the connector

The package is installed as `mysql-connector-python`, but imported in Python as `mysql.connector`.

In [ ]:
import mysql.connector
from mysql.connector import Error

print("Connector imported successfully")

## 4. Configure your MySQL connection

Change the values below to match your local MySQL setup.

- `host`: Usually `localhost` when MySQL is running on your computer.
- `port`: MySQL commonly uses `3306`. If your server uses another port, enter it here.
- `user`: Your MySQL username, often `root` in a local development setup.
- `password`: Your MySQL password.
- `database`: The database to connect to. We will create it in the next section.

**Security note:** Do not commit real passwords to Git or put them in public notebooks. For a real application, load credentials from environment variables or a secret manager.

In [ ]:
DB_CONFIG = {
    "host": "localhost",
    "port": 3306,          # Change this if your MySQL server uses another port
    "user": "root",
    "password": "YOUR_MYSQL_PASSWORD",
    # We intentionally omit "database" here because we create it first.
}

## 5. Create a database

To create a database, connect to the server without specifying a database, then execute `CREATE DATABASE`.

The `IF NOT EXISTS` clause prevents an error if the database already exists.

In [ ]:
connection = None
cursor = None

try:
    connection = mysql.connector.connect(**DB_CONFIG)
    cursor = connection.cursor()

    cursor.execute("CREATE DATABASE IF NOT EXISTS python_mysql_demo")
    print("Database is ready.")

except Error as error:
    print("MySQL error:", error)

finally:
    if cursor is not None:
        cursor.close()
    if connection is not None and connection.is_connected():
        connection.close()

## 6. Connect to the database

Now include the database name in the configuration. `mysql.connector.connect()` returns a connection object.

In [ ]:
APP_DB_CONFIG = {
    **DB_CONFIG,
    "database": "python_mysql_demo"
}

connection = mysql.connector.connect(**APP_DB_CONFIG)

print("Connected:", connection.is_connected())

connection.close()
print("Connection closed")

## 7. Connection and cursor: what is the difference?

- `connection` represents the session with the MySQL server.
- `connection.cursor()` creates a cursor.
- `cursor.execute(sql)` sends a SQL statement to MySQL.
- `cursor.fetchone()` fetches one row.
- `cursor.fetchall()` fetches all remaining rows.
- `cursor.fetchmany(n)` fetches up to `n` rows.

A cursor is not the database connection itself. Close both when you are finished.

In [ ]:
connection = mysql.connector.connect(**APP_DB_CONFIG)
cursor = connection.cursor()

cursor.execute("SELECT DATABASE()")
current_database = cursor.fetchone()

print("Current database:", current_database[0])

cursor.close()
connection.close()

## 8. Create a table

We will create an `employees` table for the examples.

The table has:
- `id`: Auto-incrementing primary key
- `name`: Employee name
- `department`: Department name
- `salary`: Decimal salary
- `is_active`: Boolean-like status stored as `TINYINT(1)`

`CREATE TABLE IF NOT EXISTS` makes this cell safe to run repeatedly.

In [ ]:
connection = mysql.connector.connect(**APP_DB_CONFIG)
cursor = connection.cursor()

create_table_sql = '''
CREATE TABLE IF NOT EXISTS employees (
    id INT AUTO_INCREMENT PRIMARY KEY,
    name VARCHAR(100) NOT NULL,
    department VARCHAR(80) NOT NULL,
    salary DECIMAL(10, 2) NOT NULL,
    is_active BOOLEAN NOT NULL DEFAULT TRUE
)
'''

cursor.execute(create_table_sql)
print("employees table is ready.")

cursor.close()
connection.close()

## 9. Insert one record — parameterized query

Use `%s` placeholders for values. Pass the values separately as a tuple.

**Do not** build SQL by concatenating strings containing user input. Parameterized queries keep data separate from SQL instructions and help prevent SQL injection.

In [ ]:
connection = mysql.connector.connect(**APP_DB_CONFIG)
cursor = connection.cursor()

insert_sql = '''
INSERT INTO employees (name, department, salary, is_active)
VALUES (%s, %s, %s, %s)
'''

employee = ("Aarav", "Engineering", 65000.00, True)

cursor.execute(insert_sql, employee)
connection.commit()

print("Inserted row ID:", cursor.lastrowid)

cursor.close()
connection.close()

## 10. Insert multiple records — `executemany()`

`executemany()` is useful when you want to execute the same parameterized statement for several rows.

In [ ]:
employees = [
    ("Diya", "Engineering", 72000.00, True),
    ("Kabir", "Human Resources", 54000.00, True),
    ("Meera", "Finance", 61000.00, False),
]

connection = mysql.connector.connect(**APP_DB_CONFIG)
cursor = connection.cursor()

insert_sql = '''
INSERT INTO employees (name, department, salary, is_active)
VALUES (%s, %s, %s, %s)
'''

cursor.executemany(insert_sql, employees)
connection.commit()

print("Rows inserted:", cursor.rowcount)

cursor.close()
connection.close()

## 11. Read records — `SELECT`

Use `fetchall()` to retrieve all rows returned by a query.

In [ ]:
connection = mysql.connector.connect(**APP_DB_CONFIG)
cursor = connection.cursor()

cursor.execute("SELECT id, name, department, salary, is_active FROM employees")
rows = cursor.fetchall()

for row in rows:
    print(row)

cursor.close()
connection.close()

## 12. Fetch one row and fetch a limited number of rows

- `fetchone()` returns one row, or `None` if no row remains.
- `fetchmany(size)` returns up to `size` rows.
- `fetchall()` returns all remaining rows.

Each fetch consumes rows from the cursor's current result.

In [ ]:
connection = mysql.connector.connect(**APP_DB_CONFIG)
cursor = connection.cursor()

cursor.execute("SELECT id, name, salary FROM employees ORDER BY id")

first_row = cursor.fetchone()
print("First row:", first_row)

next_two_rows = cursor.fetchmany(2)
print("Next two rows:", next_two_rows)

remaining_rows = cursor.fetchall()
print("Remaining rows:", remaining_rows)

cursor.close()
connection.close()

## 13. Filter and sort records

Use SQL clauses such as `WHERE`, `ORDER BY`, and `LIMIT`. Keep values parameterized.

In [ ]:
connection = mysql.connector.connect(**APP_DB_CONFIG)
cursor = connection.cursor()

minimum_salary = 60000

cursor.execute(
    '''
    SELECT id, name, department, salary
    FROM employees
    WHERE salary >= %s
    ORDER BY salary DESC
    LIMIT 10
    ''',
    (minimum_salary,)
)

for row in cursor.fetchall():
    print(row)

cursor.close()
connection.close()

## 14. Update a record

An `UPDATE` statement changes existing rows. Always use a `WHERE` clause when you intend to update only selected rows.

`rowcount` reports how many rows were affected by the last operation.

In [ ]:
connection = mysql.connector.connect(**APP_DB_CONFIG)
cursor = connection.cursor()

update_sql = '''
UPDATE employees
SET salary = %s
WHERE name = %s
'''

cursor.execute(update_sql, (70000.00, "Aarav"))
connection.commit()

print("Rows updated:", cursor.rowcount)

cursor.close()
connection.close()

## 15. Delete a record

A `DELETE` statement removes rows. Check the `WHERE` clause carefully before executing it.

In [ ]:
connection = mysql.connector.connect(**APP_DB_CONFIG)
cursor = connection.cursor()

delete_sql = "DELETE FROM employees WHERE name = %s"
cursor.execute(delete_sql, ("Kabir",))
connection.commit()

print("Rows deleted:", cursor.rowcount)

cursor.close()
connection.close()

## 16. Transactions — `commit()` and `rollback()`

A transaction groups changes into one unit of work.

- `commit()` makes the transaction's changes permanent.
- `rollback()` discards uncommitted changes.
- If an error occurs before commit, rollback is often the correct response.

The example below intentionally rolls back an insert, so the temporary row should not remain in the table.

In [ ]:
connection = mysql.connector.connect(**APP_DB_CONFIG)
cursor = connection.cursor()

try:
    cursor.execute(
        '''
        INSERT INTO employees (name, department, salary, is_active)
        VALUES (%s, %s, %s, %s)
        ''',
        ("Temporary Employee", "Testing", 30000.00, True)
    )

    # Simulate a later step failing before the transaction is committed.
    raise RuntimeError("Simulated application error")

    connection.commit()

except Exception as error:
    connection.rollback()
    print("Transaction rolled back:", error)

finally:
    cursor.close()
    connection.close()

## 17. Error handling

Connector errors can occur because the server is stopped, credentials are incorrect, the database does not exist, or a query violates a constraint.

Catch `mysql.connector.Error` to handle database-related exceptions. Avoid printing passwords or other secrets in error logs.

In [ ]:
connection = None
cursor = None

try:
    connection = mysql.connector.connect(**APP_DB_CONFIG)
    cursor = connection.cursor()

    cursor.execute("SELECT COUNT(*) FROM employees")
    print("Employee count:", cursor.fetchone()[0])

except mysql.connector.Error as error:
    print("Database operation failed:", error)

finally:
    if cursor is not None:
        cursor.close()

    if connection is not None and connection.is_connected():
        connection.close()

## 18. Reusable connection function

Instead of repeating connection settings throughout your program, create a function that returns a new connection.

This is a simple approach for small scripts. Larger applications should consider a connection pool and a clear data-access layer.

In [ ]:
def get_connection():
    return mysql.connector.connect(**APP_DB_CONFIG)


connection = None
cursor = None

try:
    connection = get_connection()
    cursor = connection.cursor()
    cursor.execute("SELECT NOW()")
    print("MySQL server time:", cursor.fetchone()[0])

finally:
    if cursor is not None:
        cursor.close()
    if connection is not None and connection.is_connected():
        connection.close()

## 19. Build reusable CRUD functions

CRUD means:

- **Create:** Insert a record
- **Read:** Retrieve records
- **Update:** Modify a record
- **Delete:** Remove a record

The functions below each open and close their own connection. They use parameterized SQL and commit write operations.

In [ ]:
def create_employee(name, department, salary):
    connection = get_connection()
    cursor = connection.cursor()

    try:
        cursor.execute(
            '''
            INSERT INTO employees (name, department, salary)
            VALUES (%s, %s, %s)
            ''',
            (name, department, salary)
        )
        connection.commit()
        return cursor.lastrowid

    except Exception:
        connection.rollback()
        raise

    finally:
        cursor.close()
        connection.close()


def get_employees():
    connection = get_connection()
    cursor = connection.cursor(dictionary=True)

    try:
        cursor.execute(
            "SELECT id, name, department, salary, is_active FROM employees ORDER BY id"
        )
        return cursor.fetchall()

    finally:
        cursor.close()
        connection.close()


def update_employee_salary(employee_id, new_salary):
    connection = get_connection()
    cursor = connection.cursor()

    try:
        cursor.execute(
            "UPDATE employees SET salary = %s WHERE id = %s",
            (new_salary, employee_id)
        )
        connection.commit()
        return cursor.rowcount

    except Exception:
        connection.rollback()
        raise

    finally:
        cursor.close()
        connection.close()


def delete_employee(employee_id):
    connection = get_connection()
    cursor = connection.cursor()

    try:
        cursor.execute(
            "DELETE FROM employees WHERE id = %s",
            (employee_id,)
        )
        connection.commit()
        return cursor.rowcount

    except Exception:
        connection.rollback()
        raise

    finally:
        cursor.close()
        connection.close()

### Try the CRUD functions

Run this cell after defining the functions. It creates a sample row, reads rows, updates the new row, and then deletes it.

In [ ]:
new_id = create_employee("Test User", "QA", 45000.00)
print("Created employee ID:", new_id)

print("\nEmployees:")
for employee in get_employees():
    print(employee)

updated = update_employee_salary(new_id, 48000.00)
print("\nRows updated:", updated)

deleted = delete_employee(new_id)
print("Rows deleted:", deleted)

## 20. Dictionary cursor

By default, rows are returned as tuples. A cursor created with `dictionary=True` returns each row as a dictionary, where column names are keys.

In [ ]:
connection = get_connection()
cursor = connection.cursor(dictionary=True)

try:
    cursor.execute("SELECT id, name, department FROM employees LIMIT 5")

    for employee in cursor.fetchall():
        print("ID:", employee["id"])
        print("Name:", employee["name"])
        print("Department:", employee["department"])
        print("---")

finally:
    cursor.close()
    connection.close()

## 21. SQL injection: unsafe vs safe code

**Unsafe pattern — never use this with untrusted input:**

```python
query = "SELECT * FROM employees WHERE name = '" + user_input + "'"
cursor.execute(query)
```

A malicious input could change the meaning of the SQL statement.

**Safe pattern — parameterized query:**

In [ ]:
user_input = "Aarav"

connection = get_connection()
cursor = connection.cursor()

try:
    cursor.execute(
        "SELECT id, name FROM employees WHERE name = %s",
        (user_input,)
    )
    print(cursor.fetchall())

finally:
    cursor.close()
    connection.close()

## 22. Environment variables for credentials

For local development, you can store credentials in environment variables instead of hardcoding them in source code.

Set variables in your operating system or development environment, then read them with Python's `os` module. Do not print secret values.

Example variable names:

- `MYSQL_HOST`
- `MYSQL_PORT`
- `MYSQL_USER`
- `MYSQL_PASSWORD`
- `MYSQL_DATABASE`

In [ ]:
import os

secure_db_config = {
    "host": os.getenv("MYSQL_HOST", "localhost"),
    "port": int(os.getenv("MYSQL_PORT", "3306")),
    "user": os.environ["MYSQL_USER"],
    "password": os.environ["MYSQL_PASSWORD"],
    "database": os.getenv("MYSQL_DATABASE", "python_mysql_demo"),
}

# Create a connection only after setting the environment variables:
# connection = mysql.connector.connect(**secure_db_config)

## 23. Common errors and troubleshooting

| Error / symptom | Possible reason | What to check |
|---|---|---|
| `ModuleNotFoundError: No module named 'mysql'` | Connector not installed in the active Python environment | Run `python -m pip install mysql-connector-python` |
| `Can't connect to MySQL server` | Server is stopped or host/port is wrong | Start MySQL and verify host and port |
| `Access denied for user` | Wrong username/password or insufficient privileges | Check credentials and grants |
| `Unknown database` | Database does not exist or name is misspelled | Create the database or correct its name |
| `Table doesn't exist` | Table has not been created in the selected database | Run the `CREATE TABLE` cell |
| `Not all parameters were used` | Placeholder count and supplied values do not match | Match every `%s` with one value |
| Changes are missing | Transaction was not committed | Call `connection.commit()` after successful writes |

### Check which Python is running

If installation succeeds but import fails, your notebook kernel may be using a different Python interpreter.

In [ ]:
import sys

print("Python executable:", sys.executable)
print("Python version:", sys.version)

## 24. Important methods — quick revision

| Method / property | Meaning |
|---|---|
| `mysql.connector.connect()` | Open a MySQL connection |
| `connection.is_connected()` | Check whether the connection is active |
| `connection.cursor()` | Create a cursor |
| `cursor.execute()` | Execute one SQL statement |
| `cursor.executemany()` | Execute a statement for multiple parameter sets |
| `cursor.fetchone()` | Fetch one row |
| `cursor.fetchmany(n)` | Fetch up to `n` rows |
| `cursor.fetchall()` | Fetch all remaining rows |
| `cursor.rowcount` | Number of rows affected / returned, depending on operation |
| `cursor.lastrowid` | Auto-generated ID from the last insert, when available |
| `connection.commit()` | Commit transaction changes |
| `connection.rollback()` | Roll back uncommitted transaction changes |
| `cursor.close()` | Close cursor |
| `connection.close()` | Close connection |

## 25. Best practices

1. Use parameterized queries (`%s`) for values.
2. Never commit passwords or secrets to Git.
3. Close cursors and connections using `finally` or context managers.
4. Commit successful write transactions.
5. Roll back a transaction when a write operation fails.
6. Use least-privilege MySQL accounts instead of using `root` in production.
7. Validate input in your application, but still parameterize SQL.
8. Use connection pooling for applications with many database operations.
9. Keep SQL and database access code organized rather than putting everything in one large file.
10. Do not share one cursor concurrently across unrelated tasks.

## 26. Practice exercises

Try these without looking at the earlier examples:

1. Create a `students` table with `id`, `name`, `course`, and `marks`.
2. Insert five students using `executemany()`.
3. Display students who scored more than 75.
4. Update one student's marks using their ID.
5. Delete students whose marks are below 35.
6. Count students in each course using `GROUP BY`.
7. Use a transaction to insert two related records and roll back if the second insert fails.
8. Write reusable functions for all four CRUD operations.

## 27. Final mental model

Remember this sequence:

```python
# 1. Connect
connection = mysql.connector.connect(...)

# 2. Create cursor
cursor = connection.cursor()

# 3. Execute SQL
cursor.execute("SELECT * FROM employees")

# 4. Fetch result
rows = cursor.fetchall()

# 5. Commit when you perform successful writes
connection.commit()

# 6. Close resources
cursor.close()
connection.close()
```

**Core idea:** `Connection` manages the database session, `Cursor` executes SQL and reads results, and `commit()` / `rollback()` control transaction outcomes.